In [3]:
def ensure_list(input_item):
    """Ensure that item is a list, generally for iterability."""
    if not isinstance(input_item, list):
        return [input_item]
    return input_item


def split_request(
    full_request,  # User request
    this_values,  # key: [values] for the adaptor component
    **config,
):
    """
    Basic request splitter, splits based on whether the values are relevant to
    the specific adaptor.
    More complex constraints may need a more detailed splitter.
    """
    this_request = {}
    # loop over keys in this_values, i.e. the keys relevant to this_adaptor
    for key in list(this_values):
        # get request values for that key
        req_vals = full_request.get(key, [])
        # filter for values relevant to this_adaptor:
        these_vals = [
            v for v in ensure_list(req_vals) if v in this_values.get(key, [])
        ]
        if len(these_vals) > 0:
            # if values then add to request
            this_request[key] = these_vals
        elif key not in config.get("optional_keys", []):
            # If not an optional key, then return an empty dictionary.
            #  optional keys must be set in the adaptor.json via gecko
            return {}

    return this_request

In [8]:
request = {
    'hday': ['07'], 'time': ['00:00'], 'hyear': ['2023'], 'hmonth': ['05'],
    'variable': ['river_discharge_in_the_last_6_hours'], 'model_levels': 'surface_level',
    'system_version': 'version_4_0'
}

# url
url_values = {
    'variable': ['elevation', 'field_capacity', 'wilting_point', 'upstream_area', 'soil_depth'],
    'soil_level': ['1', '2', '3'],
    'model_levels': ['soil_levels', 'surface_level'],
    'system_version': ['version_5_0', 'version_4_0', 'version_3_5', 'version_3_0','version_2_0']
}
url_optional_keys = []
this_request= {}

# MARS

mars_values = {
    'hday': ['01', '02', '03', '04', '05', '06', '07', '08', '09', '10', '11', '12', '13', '14', '15', '16', '17', '18', '19', '20', '21', '22', '23', '24', '25', '26', '27', '28', '29', '30', '31'],
    'time': ['00:00', '06:00', '12:00', '18:00'],
    'hyear': [
        '1991', '1992', '1993', '1994', '1995', '1996', '1997', '1998', '1999', '2000', '2001', '2002', '2003', '2004', '2005', '2006', '2007', '2008', '2009', '2010', '2011', '2012', '2013', '2014', '2015', '2016', '2017', '2018', '2019', '2020', '2021', '2022', '2023'
    ],
    'hmonth': ['01', '02', '03', '04', '05', '06', '07', '08', '09', '10', '11', '12'],
    'variable': [
        'river_discharge_in_the_last_6_hours', 'river_discharge_in_the_last_24_hours', 'snow_depth_water_equivalent',
        'volumetric_soil_moisture'
    ],
    'soil_level': ['', '1', '2', '3'],
    'model_levels': ['surface_level', 'soil_levels'],
    'system_version': [
        'version_4_0', 'version_3_5', 'version_3_0','version_2_0'
    ]
}
mars_optional_keys = ['soil_level']
this_request = {} 


In [9]:
split_request(
    request,  # User request
    mars_values,  # key: [values] for the adaptor component
    optional_keys = mars_optional_keys
)

{'hday': ['07'],
 'time': ['00:00'],
 'hyear': ['2023'],
 'hmonth': ['05'],
 'variable': ['river_discharge_in_the_last_6_hours'],
 'model_levels': ['surface_level'],
 'system_version': ['version_4_0']}

In [7]:
full_request = request
this_values = mars_values
this_request = {}
# loop over keys in this_values, i.e. the keys relevant to this_adaptor
for key in list(this_values):
    print(key)
    # get request values for that key
    req_vals = full_request.get(key, [])
    # filter for values relevant to this_adaptor:
    these_vals = [
        v for v in ensure_list(req_vals) if v in this_values.get(key, [])
    ]
    if len(these_vals) > 0:
        # if values then add to request
        this_request[key] = these_vals
    elif key not in []:
        print("not-optional")
        # If not an optional key, then return an empty dictionary.
        #  optional keys must be set in the adaptor.json via gecko
        this_request = {}
        break

this_request

hday
time
hyear
hmonth
variable
soil_level
not-optional


{}

In [1]:
from cads_adaptors import mapping

req_mapping = {"options":{"wants_dates": True}}
request = {
    "year": "2003",
    "month": "03",
    "day": "03",
    "hyear": "2004",
    "hmonth": "04",
    "hday": "04",
}
mapped_request = mapping.apply_mapping(request, req_mapping)
mapped_request

{'date': ['2003-03-03'], 'hdate': ['2004-04-04']}

In [1]:
from cads_adaptors import mapping

req_mapping = {"options":{"wants_dates": True}}
request = {
    "hyear": "2003",
    "hmonth": "03",
    "hday": "03",
}
mapped_request = mapping.apply_mapping(request, req_mapping)
mapped_request
# assert "hdate" in mapped_request
# assert mapped_request["hdate"][0] == f"{request['hyear']}-{request['hmonth']}-{request['hday']}"


{'hdate': ['2003-03-03']}

In [1]:
from cads_adaptors.constraints import apply_constraints_in_old_cds_fashion, apply_constraints_alternate

In [2]:
constraints = [
{"spatial_aggregation": ["maritime_country_level", "maritime_sub_country_level"], "temporal_aggregation": ["daily", "hourly", "monthly", "seasonal", "annual"], "variable": ["wind_speed_at_100m", "wind_speed_at_10m"]},
{"spatial_aggregation": ["country_level", "sub_country_level"], "temporal_aggregation": ["daily", "hourly", "monthly", "seasonal", "annual"], "variable": ["wind_speed_at_100m", "wind_speed_at_10m", "surface_downwelling_shortwave_radiation", "pressure_at_sea_level", "2m_air_temperature", "total_precipitation"]},
{"month": ["01"], "spatial_aggregation": ["original_grid"], "temporal_aggregation": ["hourly"], "variable": ["wind_speed_at_100m", "wind_speed_at_10m", "pressure_at_sea_level", "2m_air_temperature", "total_precipitation"], "year": ["1979", "1980", "1981", "1982", "1983", "1984", "1985", "1986", "1987", "1988", "1989", "1990", "1991", "1992", "1993", "1994", "1995", "1996", "1997", "1998", "1999", "2000", "2001", "2002", "2003", "2004", "2005", "2006", "2007", "2008", "2009", "2010", "2011", "2012", "2013", "2014", "2015", "2016", "2017", "2018", "2019", "2020", "2021", "2022", "2023"]},
{"month": ["01", "02", "03", "04", "05"], "spatial_aggregation": ["original_grid"], "temporal_aggregation": ["hourly"], "variable": ["surface_downwelling_shortwave_radiation"], "year": ["1979", "1980", "1981", "1982", "1983", "1984", "1985", "1986", "1987", "1988", "1989", "1990", "1991", "1992", "1993", "1994", "1995", "1996", "1997", "1998", "1999", "2000", "2001", "2002", "2003", "2004", "2005", "2006", "2007", "2008", "2009", "2010", "2011", "2012", "2013", "2014", "2015", "2016", "2017", "2018", "2019", "2020", "2021", "2022", "2023"]},
{"month": ["02", "03", "04", "05"], "spatial_aggregation": ["original_grid"], "temporal_aggregation": ["hourly"], "variable": ["wind_speed_at_100m", "wind_speed_at_10m", "pressure_at_sea_level", "2m_air_temperature", "total_precipitation"], "year": ["2019", "2020", "2021", "2022", "2023"]},
{"month": ["06", "07", "08", "09", "10", "11", "12"], "spatial_aggregation": ["original_grid"], "temporal_aggregation": ["hourly"], "variable": ["wind_speed_at_100m", "wind_speed_at_10m", "pressure_at_sea_level", "2m_air_temperature", "total_precipitation"], "year": ["2019", "2020", "2021", "2022"]},
{"month": ["06", "07", "08", "09", "10", "11", "12"], "spatial_aggregation": ["original_grid"], "temporal_aggregation": ["hourly"], "variable": ["surface_downwelling_shortwave_radiation"], "year": ["1979", "1980", "1981", "1982", "1983", "1984", "1985", "1986", "1987", "1988", "1989", "1990", "1991", "1992", "1993", "1994", "1995", "1996", "1997", "1998", "1999", "2000", "2001", "2002", "2003", "2004", "2005", "2006", "2007", "2008", "2009", "2010", "2011", "2012", "2013", "2014", "2015", "2016", "2017", "2018", "2019", "2020", "2021", "2022"]},
{"energy_product_type": ["capacity_factor_ratio"], "spatial_aggregation": ["maritime_country_level", "maritime_sub_country_level"], "temporal_aggregation": ["hourly"], "variable": ["wind_power_generation_offshore"]},
{"energy_product_type": ["capacity_factor_ratio"], "spatial_aggregation": ["country_level"], "temporal_aggregation": ["daily"], "variable": ["hydro_power_generation_reservoirs", "hydro_power_generation_rivers"]},
{"energy_product_type": ["capacity_factor_ratio"], "spatial_aggregation": ["country_level", "sub_country_level"], "temporal_aggregation": ["hourly"], "variable": ["solar_photovoltaic_power_generation", "wind_power_generation_onshore"]},
{"energy_product_type": ["capacity_factor_ratio"], "month": ["01"], "spatial_aggregation": ["original_grid"], "temporal_aggregation": ["hourly"], "variable": ["solar_photovoltaic_power_generation", "wind_power_generation_offshore", "wind_power_generation_onshore"], "year": ["1979", "1980", "1981", "1982", "1983", "1984", "1985", "1986", "1987", "1988", "1989", "1990", "1991", "1992", "1993", "1994", "1995", "1996", "1997", "1998", "1999", "2000", "2001", "2002", "2003", "2004", "2005", "2006", "2007", "2008", "2009", "2010", "2011", "2012", "2013", "2014", "2015", "2016", "2017", "2018", "2019", "2020", "2021", "2022", "2023"]},
{"energy_product_type": ["capacity_factor_ratio"], "month": ["02", "03", "04", "05"], "spatial_aggregation": ["original_grid"], "temporal_aggregation": ["hourly"], "variable": ["solar_photovoltaic_power_generation", "wind_power_generation_offshore", "wind_power_generation_onshore"], "year": ["2019", "2020", "2021", "2022", "2023"]},
{"energy_product_type": ["capacity_factor_ratio"], "month": ["06", "07", "08", "09", "10", "11", "12"], "spatial_aggregation": ["original_grid"], "temporal_aggregation": ["hourly"], "variable": ["solar_photovoltaic_power_generation", "wind_power_generation_offshore", "wind_power_generation_onshore"], "year": ["2019", "2020", "2021", "2022"]},
{"energy_product_type": ["energy", "power"], "spatial_aggregation": ["country_level"], "temporal_aggregation": ["daily"], "variable": ["electricity_demand", "hydro_power_generation_reservoirs", "hydro_power_generation_rivers"]},
{"energy_product_type": ["energy", "power"], "spatial_aggregation": ["country_level"], "temporal_aggregation": ["hourly"], "variable": ["solar_photovoltaic_power_generation"]}
]
constraints = [{k: set(v) for k,v in constraint.items()} for constraint in constraints]
constraints

[{'spatial_aggregation': {'maritime_country_level',
   'maritime_sub_country_level'},
  'temporal_aggregation': {'annual', 'daily', 'hourly', 'monthly', 'seasonal'},
  'variable': {'wind_speed_at_100m', 'wind_speed_at_10m'}},
 {'spatial_aggregation': {'country_level', 'sub_country_level'},
  'temporal_aggregation': {'annual', 'daily', 'hourly', 'monthly', 'seasonal'},
  'variable': {'2m_air_temperature',
   'pressure_at_sea_level',
   'surface_downwelling_shortwave_radiation',
   'total_precipitation',
   'wind_speed_at_100m',
   'wind_speed_at_10m'}},
 {'month': {'01'},
  'spatial_aggregation': {'original_grid'},
  'temporal_aggregation': {'hourly'},
  'variable': {'2m_air_temperature',
   'pressure_at_sea_level',
   'total_precipitation',
   'wind_speed_at_100m',
   'wind_speed_at_10m'},
  'year': {'1979',
   '1980',
   '1981',
   '1982',
   '1983',
   '1984',
   '1985',
   '1986',
   '1987',
   '1988',
   '1989',
   '1990',
   '1991',
   '1992',
   '1993',
   '1994',
   '1995',
   '

In [10]:
selection_1 = {
    'variable': {"wind_speed_at_10m"}
}
selection_2 = {
    'variable': {"wind_speed_at_10m", "hydro_power_generation_reservoirs"}
}
selection_3 = {
    'variable': {"wind_speed_at_10m", "hydro_power_generation_reservoirs"},
    "energy_product_type": {"capacity_factor_ratio"}
}
selection_4 = {
    'variable': {"wind_speed_at_10m", "hydro_power_generation_reservoirs"},
    "energy_product_type": {"capacity_factor_ratio"},
    "month": {"12"},
}
selection_5 = {
    'variable': {"wind_speed_at_10m", "hydro_power_generation_reservoirs"},
    "energy_product_type": {"capacity_factor_ratio"},
    "temporal_aggregation": {"daily"},
}
selections = [
    selection_1, selection_2, selection_3, selection_4, selection_5
]

In [11]:
# constrained_form_1 = apply_constraints_in_old_cds_fashion({}, selection_1, constraints)
# constrained_form_2 = apply_constraints_in_old_cds_fashion({}, selection_2, constraints)
# constrained_form_3 = apply_constraints_in_old_cds_fashion({}, selection_3, constraints)
# constrained_form_4 = apply_constraints_in_old_cds_fashion({}, selection_4, constraints)
constrained_form_5 = apply_constraints_in_old_cds_fashion({}, selection_5, constraints)

In [12]:
constrained_form_5_b = apply_constraints_alternate({}, selection_5, constraints)


In [13]:
for i, selection in enumerate(selections):
    print(
        i, 
        apply_constraints_in_old_cds_fashion({}, selection_5, constraints) ==
        apply_constraints_alternate({}, selection_5, constraints)
    )



0 True
1 True
2 True
3 True
4 True
